<a href="https://colab.research.google.com/github/C-1Blue/ds2002-fa26/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:


pop_tarts = cat[cat['product_name'].str.lower().str.contains('pop')]

print(f"Number of Pop-Tart rows: {len(pop_tarts)}")
print("\nMatching rows:")
print(pop_tarts)

Number of Pop-Tart rows: 3

Matching rows:
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:

strawberry_pop_tarts = pop_tarts[pop_tarts['product_name'].str.lower().str.contains('straw')]
non_strawberry_pop_tarts = pop_tarts[~pop_tarts['product_name'].str.lower().str.contains('straw')]

print("Strawberry Pop-Tarts:")
display(strawberry_pop_tarts[['sku', 'product_name']])

print("\nNon-Strawberry Pop-Tarts:")
display(non_strawberry_pop_tarts[['sku', 'product_name']])

Strawberry Pop-Tarts:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry



Non-Strawberry Pop-Tarts:


,sku,product_name


**Pop-Tart rows that are a different product:**

Looking at our filtered dataset, all 3 matching 'Pop-Tart' rows are strawberry flavors:
1. `PT-12`: pop-tarts strawberry
2. `SKU#459812`: Strawberry Pop-Tart
3. `POPTART-STRAW`: Pop-Tarts Strawberry

There are **0** Pop-Tart rows that are a different flavor (non-strawberry) in this dataset.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
sku_mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

# Map the SKUs and use .fillna(cat['sku']) to keep unmapped SKUs as themselves
cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])


display(cat)

,sku,product_name,category,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,PT-12
1,SKU#459812,Strawberry Pop-Tart,Breakfast,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# Show counts before normalization
print("Before normalization:")
print(cat['category'].value_counts())

# Apply strip and title-case
cat['category_normalized'] = cat['category'].str.strip().str.title()

# Show counts after normalization
print("\nAfter normalization:")
print(cat['category_normalized'].value_counts())



Before normalization:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

After normalization:
category_normalized
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import string

def normalize_name(name):
    if not isinstance(name, str):
        return ""
    name_lower = name.lower()
    clean_name = name_lower.translate(str.maketrans('', '', string.punctuation))
    words_sorted = sorted(clean_name.split())
    return " ".join(words_sorted)

candidates = cat.copy()
candidates['normalized_name'] = candidates['product_name'].apply(normalize_name)

grouped = candidates.groupby('normalized_name').agg(
    skus=('sku', list),
    sku_count=('sku', 'count'),
    original_names=('product_name', list)
).reset_index()

fragmented_groups = grouped[grouped['sku_count'] > 1]

display(fragmented_groups)

,normalized_name,skus,sku_count,original_names
1,poptarts strawberry,"[PT-12, POPTART-STRAW]",2,"[pop-tarts strawberry, Pop-Tarts Strawberry]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# Assert no nulls in the canonical column
assert cat['canonical_sku'].notnull().all(), "Error: Found null values in the canonical_sku column!"

# Assert that the number of rows did not change

assert len(cat) == 3, f"Error: Row count changed! Expected 3, got {len(cat)}"

print("Assertions passed successfully! The mapping is safe.")

Assertions passed successfully! The mapping is safe.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

### **a) Fragmented Product Families Beyond Pop-Tarts**


Carbonated Soft Drinks: Items like "Coca-Cola 12oz Can", "Coke 12-pack", and "Coke Classic" represent the same  brand and flavor. We would group them by creating a canonical brand mapping (`Coke` / `Coca-Cola`) and clustering them by base flavor and package type.
Pain Relievers:
 Brand names like "Advil 200mg", "Motrin IB", and generic "Ibuprofen 200mg" contain the exact same active ingredient. We would group them under a canonical ingredient classification (`Ibuprofen, 200mg Tablet`).

### **b) Data Cleaning vs. Business Decisions**

Consolidation shifts from a technical "data-cleaning" task to a **business decision** when merging two SKUs changes how inventory is tracked, how sales are reported, or how products are priced.

* **Example of Uncertainty:** Consider **"Diet Coke"** versus **"Coke Zero Sugar"**. A data scientist might see them as both being "sugar-free colas" and group them. However, they use different sweetener formulations, have completely different branding, and target different customer demographics. Bundling them under a single canonical SKU would corrupt sales velocity data and inventory replenishment cycles.
 I would ask the Category Manager or Brand Merchandiser
 responsible for beverages, as they understand the customer demand patterns and supply-chain logistics for these specific products.